[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/ProyectoApophis/Main.ipynb)

---
## 1. Marco Teórico

### 1.1 El Baricentro del Sistema Solar (SSB)

El **Baricentro del Sistema Solar** (Solar System Barycenter, SSB) es el centro de masa de todo el Sistema Solar. Si denotamos la masa del cuerpo $i$ como $m_i$ y su posición como $\mathbf{r}_i$, el SSB se define como:

$$\mathbf{R}_{\rm SSB} = \frac{\sum_i m_i \, \mathbf{r}_i}{\sum_i m_i}$$

Por definición, en el **marco del SSB** el **momento lineal total** del Sistema Solar es cero:

$$\mathbf{P}_{\rm total} = \sum_i m_i \, \mathbf{v}_i = \mathbf{0}$$

Esto lo convierte en un **sistema de referencia inercial** (o casi inercial para propósitos prácticos), lo que significa que las leyes de Newton se aplican directamente sin términos de arrastre ficticios.

| Marco de referencia | Origen | Inercial | Usado por |
|---|---|---|---|
| **SSB eclíptico J2000** | Baricentro del SS | Sí | JPL DE441, SPICE |
| Heliocéntrico eclíptico J2000 | Centro del Sol | $\approx$ Sí (el Sol oscila ~0.01 AU) | Astronomía clásica |
| Geocéntrico | Centro de la Tierra | No (acelerado) | Satélites, observaciones |


### 1.2 El eje eclíptico J2000

Las posiciones y velocidades en el SSB se expresan en el sistema de coordenadas **eclíptico J2000**:
- **Eje X**: apunta hacia el equinoccio vernal del año 2000.0.
- **Eje Z**: perpendicular al plano de la eclíptica (plano de la órbita terrestre).
- **Eje Y**: completa la base derecha $\hat{z} = \hat{x} \times \hat{y}$.

Este es el sistema que usa JPL Horizons por defecto al pedir vectores de estado (`COORD_TYPE=2`).

---

### 1.3 Vectores de Estado: Posición y Velocidad

El **vector de estado** de un cuerpo en el SSB tiene 6 componentes:

$$\mathbf{y}_i = (x_i, y_i, z_i, \dot{x}_i, \dot{y}_i, \dot{z}_i) = (\mathbf{r}_i, \mathbf{v}_i)$$

donde:
- $(x_i, y_i, z_i)$ es la posición en Unidades Astronómicas (UA).
- $(\dot{x}_i, \dot{y}_i, \dot{z}_i)$ es la velocidad en UA/día.

JPL Horizons proporciona estos datos para cualquier cuerpo del Sistema Solar a partir de las **efemérides DE441** (integración numérica de alta precisión).

---

### 1.4 El Parámetro Gravitacional $\mu = GM$

La constante gravitacional universal $G = 6.674 \times 10^{-11}$ m³ kg⁻¹ s⁻² se conoce con una precisión relativa de ~22 ppm, mientras que el **parámetro gravitacional** $\mu_i = GM_i$ se conoce con muchísima mayor precisión (hasta 11 dígitos significativos) mediante observaciones orbitales de naves espaciales.

Por esta razón, en astrodinámica se trabaja directamente con $\mu$ en lugar de $G$ y $M$ por separado:

| Cuerpo | $\mu = GM$ (m³/s²) | $\mu$ (UA³/día²) | $m/M_\odot$ |
|--------|---------------------|------------------|---------------|
| **Sol** | $1.32712 \times 10^{20}$ | $2.9592 \times 10^{-4}$ | $1.0$ |
| **Tierra** | $3.9860 \times 10^{14}$ | $8.8878 \times 10^{-10}$ | $3.003 \times 10^{-6}$ |
| **Luna** | $4.9048 \times 10^{12}$ | $1.0931 \times 10^{-11}$ | $3.694 \times 10^{-8}$ |
| **Apophis** | $\approx 1.8 \times 10^{0}$ | $\approx 4 \times 10^{-24}$ | $\approx 1.4 \times 10^{-20}$ |

---

### 1.5 El Problema de N-Cuerpos: Ecuaciones de Movimiento

Para un sistema de $N$ cuerpos con masas $m_i$ y posiciones $\mathbf{r}_i$ respecto al SSB, las ecuaciones de movimiento de Newton son:

$$\boxed{\ddot{\mathbf{r}}_i = \sum_{j \neq i} \frac{G m_j (\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}}$$

En nuestra simulación tenemos $N = 4$ cuerpos: Sol, Tierra, Luna y Apophis. Esto nos da un sistema de $4 \times 3 = 12$ EDOs de segundo orden, o equivalentemente $24$ EDOs de primer orden.

**Constantes de movimiento:**
- Energía mecánica total: $E = \frac{1}{2}\sum_i m_i v_i^2 - \sum_{i<j} \frac{G m_i m_j}{|\mathbf{r}_i - \mathbf{r}_j|}$
- Momento lineal total: $\mathbf{P} = \sum_i m_i \mathbf{v}_i = \mathbf{0}$ (en el SSB)
- Momento angular total: $\mathbf{L} = \sum_i m_i (\mathbf{r}_i \times \mathbf{v}_i)$



In [1]:
!pip install celluloid astropy pymcel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.0 MB/s eta 0:00:00


In [16]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
from celluloid import Camera
import pandas as pd

In [17]:
# Constantes de conversión
AU_m   = pc.constantes.au      # 1 UA en metros
day_s  = pc.constantes.día     # 1 día en segundos
yr_d   = pc.constantes.yr / pc.constantes.día  # 1 año en días
G_SI   = pc.constantes.G       # Constante gravitacional [m^3 kg^-1 s^-2]
c_luz  = pc.constantes.c       # Velocidad de la luz [m/s]

# Factor de conversión de mu: m^3/s^2 -> UA^3/día^2
fac_mu = day_s**2 / AU_m**3

In [18]:
# Constantes gravitacionales (SI) m^3/s^2
mu_sun_SI      = pc.constantes.mu_sun
mu_mercury_SI  = pc.constantes.mu_mercury
mu_venus_SI    = pc.constantes.mu_venus
mu_earth_SI    = pc.constantes.mu_earth
mu_moon_SI     = pc.constantes.mu_moon
mu_mars_SI     = pc.constantes.mu_mars
mu_jupiter_SI  = pc.constantes.mu_jupiter
mu_saturn_SI   = pc.constantes.mu_saturn
mu_uranus_SI   = pc.constantes.mu_uranus
mu_neptune_SI  = pc.constantes.mu_neptune

m_apophis      = 2.7e10                          # kg
mu_apophis_SI  = G_SI * m_apophis                # m^3/s^2

# Construcción del DataFrame
datos_mu = [
    ('Sol',       mu_sun_SI,     mu_sun_SI * fac_mu,     1.0),
    ('Mercurio',  mu_mercury_SI, mu_mercury_SI * fac_mu, mu_mercury_SI / mu_sun_SI),
    ('Venus',     mu_venus_SI,   mu_venus_SI * fac_mu,   mu_venus_SI / mu_sun_SI),
    ('Tierra',    mu_earth_SI,   mu_earth_SI * fac_mu,   mu_earth_SI / mu_sun_SI),
    ('Luna',      mu_moon_SI,    mu_moon_SI * fac_mu,    mu_moon_SI / mu_sun_SI),
    ('Marte',     mu_mars_SI,    mu_mars_SI * fac_mu,    mu_mars_SI / mu_sun_SI),
    ('Jupiter',   mu_jupiter_SI, mu_jupiter_SI * fac_mu, mu_jupiter_SI / mu_sun_SI),
    ('Saturno',   mu_saturn_SI,  mu_saturn_SI * fac_mu,  mu_saturn_SI / mu_sun_SI),
    ('Urano',     mu_uranus_SI,  mu_uranus_SI * fac_mu,  mu_uranus_SI / mu_sun_SI),
    ('Neptuno',   mu_neptune_SI, mu_neptune_SI * fac_mu, mu_neptune_SI / mu_sun_SI),
    ('Apophis',   mu_apophis_SI, mu_apophis_SI * fac_mu, mu_apophis_SI / mu_sun_SI),
]

cuerpos_mu = pd.DataFrame(datos_mu, columns=['Cuerpo', 'mu (m^3/s^2)', 'mu (UA^3/d^2)', 'm/M_sol'])
cuerpos_mu

,Cuerpo,mu (m^3/s^2),mu (UA^3/d^2),m/M_sol
0,Sol,1.327124e+20,2.959122e-04,1.000000e+00
1,Mercurio,2.203187e+13,4.912500e-11,1.660121e-07
2,Venus,3.248586e+14,7.243452e-10,2.447838e-06
3,Tierra,3.986004e+14,8.887692e-10,3.003490e-06
4,Luna,4.902800e+12,1.093189e-11,3.694303e-08
5,Marte,4.282838e+13,9.549549e-11,3.227156e-07
6,Jupiter,1.267128e+17,2.825346e-07,9.547919e-04
7,Saturno,3.794058e+16,8.459706e-08,2.858857e-04
8,Urano,5.794556e+15,1.292027e-08,4.366250e-05
9,Neptuno,6.836527e+15,1.524357e-08,5.151384e-05


In [20]:
CUERPOS = [
    ('Sol',     '10',    'majorbody', mu_sun_SI),
    ('Tierra',  '399',   'majorbody', mu_earth_SI),
    ('Luna',    '301',   'majorbody', mu_moon_SI),
    ('Apophis', '99942', 'smallbody', mu_apophis_SI),
]

In [13]:
def Estados(cuerpos, propiedades, epochs):
    VectorEstadoSI = {}
    for nombre, id in cuerpos:
        _, _, salida = pc.consulta_horizons(
            id=id,
            location="@0",
            datos="vectors",
            propiedades=propiedades,
            epochs=epochs,
        )
        VectorEstadoSI[nombre] = np.array(salida, dtype=float).reshape(-1)
    return VectorEstadoSI

def UnidadesCanonicas(EstadosSI, MasasSI, UL=1.495978707e11, UM=1.98847e30):

    UL, UM, UT, Gc = pc.unidades_canonicas(UL=UL, UM=UM)
    vel_unit = UL / UT

    EstadoCanonico = {}
    for nombre, vec in EstadosSI.items():
        x, y, z, vx, vy, vz = vec
        EstadoCanonico[nombre] = np.array([
            x / UL,
            y / UL,
            z / UL,
            vx / vel_unit,
            vy / vel_unit,
            vz / vel_unit,
        ])

    MasasCanonico = {nombre: MasasSI[nombre] / UM for nombre in MasasSI}
    return EstadoCanonico, MasasCanonico, UL, UM, UT, Gc

In [27]:
#epochs_ini = {'start': '2026-04-11', 'stop': '2026-04-12', 'step': '1d'}
epochs_ini = "2026-04-11"

In [28]:
def Estados(cuerpos, propiedades, epochs):
    VectorEstadoSI = {}
    for nombre, id in cuerpos:
        _, _, salida = pc.consulta_horizons(
            id=id,
            location="@0",
            datos="vectors",
            propiedades=propiedades,
            epochs=epochs,
        )
        # Convertir la salida a un vector único [x, y, z, vx, vy, vz]
        VectorEstadoSI[nombre] = np.array(salida, dtype=float).reshape(-1)
    return VectorEstadoSI

def UnidadesCanonicas(EstadosSI, MasasSI, UL=1.495978707e11, UM=1.98847e30):
    UL, UM, UT, Gc = pc.unidades_canonicas(UL=UL, UM=UM)
    vel_unit = UL / UT

    EstadoCanonico = {}
    for nombre, vec in EstadosSI.items():
        # Asegurarse de que el vector tenga 6 elementos: [x, y, z, vx, vy, vz]
        if len(vec) != 6:
            raise ValueError(f"El vector de estado para {nombre} debe tener 6 elementos: [x, y, z, vx, vy, vz]")
        x, y, z, vx, vy, vz = vec
        EstadoCanonico[nombre] = np.array([
            x / UL,
            y / UL,
            z / UL,
            vx / vel_unit,
            vy / vel_unit,
            vz / vel_unit,
        ])

    MasasCanonico = {nombre: MasasSI[nombre] / UM for nombre in MasasSI}
    return EstadoCanonico, MasasCanonico, UL, UM, UT, Gc

# --- Adaptación del segundo script ---
estados_ini = {}  # Diccionario: nombre -> [x, y, z, vx, vy, vz]

for nombre, horizons_id, id_type, mu in CUERPOS:
    print(f'Consultando Horizons para {nombre} (ID={horizons_id})...')
    try:
        resultado = pc.consulta_horizons(
            id=horizons_id,
            location='@0',
            epochs=epochs_ini,
            datos='vectors',
        )
        print(f"Tipo de resultado: {type(resultado)}")
        print(f"Contenido: {resultado}")  # Ver qué devuelve
        break  # Solo probar el primero para no saturar la salida
    except Exception as e:
        print(f"Error: {e}")

# --- Uso de las funciones ---
# Ejemplo de cómo usar las funciones con los datos adaptados
MasasSI = {nombre: mu for nombre, _, _, mu in CUERPOS}  # Diccionario de masas
EstadosSI = estados_ini  # Ya está en el formato correcto: [x, y, z, vx, vy, vz]

# Convertir a unidades canónicas
EstadoCanonico, MasasCanonico, UL, UM, UT, Gc = UnidadesCanonicas(EstadosSI, MasasSI)

Consultando Horizons para Sol (ID=10)...
Tipo de resultado: <class 'tuple'>
Contenido: (<Table masked=True length=1>
targetname datetime_jd ...        range               range_rate      
   ---          d      ...          AU                  AU / d        
   str8      float64   ...       float64               float64        
---------- ----------- ... -------------------- ----------------------
  Sun (10)   2461141.5 ... 0.005956811816482579 -3.805858052900413e-06, np.float64(2461141.5), array([-3.53325160e+08, -8.17897981e+08,  1.76187615e+07,  1.19440312e+01,
        2.01470051e+00, -2.43515046e-01]))


In [29]:
import numpy as np
import pymcel as pc

CUERPOS = [
    ('Sol',     '10',    'majorbody', mu_sun_SI),
    ('Tierra',  '399',   'majorbody', mu_earth_SI),
    ('Luna',    '301',   'majorbody', mu_moon_SI),
    ('Apophis', '99942', 'smallbody', mu_apophis_SI),
]

epochs_ini = '2026-04-11'  # Solo una fecha

estados_ini = {}

for nombre, horizons_id, id_type, mu in CUERPOS:
    print(f'Consultando Horizons para {nombre} (ID={horizons_id})...')
    try:
        # pc.consulta_horizons devuelve una tupla: (tabla, tiempo_jd, vector_estado)
        tabla, tiempo_jd, vector_estado = pc.consulta_horizons(
            id=horizons_id,
            location='@0',
            epochs=epochs_ini,
            datos='vectors',
        )
        # El vector de estado ya está en el formato [x, y, z, vx, vy, vz]
        estados_ini[nombre] = vector_estado
    except Exception as e:
        print(f"Error al consultar {nombre}: {e}")

# Verificar el resultado
for nombre, vector in estados_ini.items():
    print(f"{nombre}: {vector}")

Consultando Horizons para Sol (ID=10)...
Consultando Horizons para Tierra (ID=399)...
Consultando Horizons para Luna (ID=301)...
Consultando Horizons para Apophis (ID=99942)...
Sol: [-3.53325160e+08 -8.17897981e+08  1.76187615e+07  1.19440312e+01
  2.01470051e+00 -2.43515046e-01]
Tierra: [-1.40521419e+11 -5.39341377e+10  2.18549253e+07  1.00748836e+04
 -2.79712608e+04  5.00676680e-01]
Luna: [-1.40325271e+11 -5.42785809e+10 -2.09805396e+05  1.09052134e+04
 -2.74456191e+04  7.29104911e+01]
Apophis: [-2.38689434e+10  1.47962908e+11 -8.47922873e+09 -2.82188454e+04
  2.98120026e+02 -6.83682043e+02]
